# Lab 10–11: ML Pipelines & Experiment Tracking

**Z5008 Big Data Lab (MTech Core) · IIT Madras Zanzibar**

Leakage-free feature scaling pipelines and hyperparameter metric logging.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5008/labs/BD04_Distributed_ML_Pipelines_MLlib.ipynb)

Run cells top to bottom (`Runtime > Run all`). Spark steps include pure-Python fallbacks when executed outside a Spark/JVM container.

## 1. Leakage-Free ML Pipeline & MLflow-Style Run Tracking

Chaining `StandardScaler` and `LogisticRegression` inside `Pipeline` guarantees scaler statistics never leak from validation/test splits while logging comparable hyperparameters and metrics.

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(5008)
X = np.vstack([rng.normal([0, 0, 1], 1.0, (320, 3)), rng.normal([1.7, 1.5, 2.2], 0.9, (80, 3))])
y = np.array([0] * 320 + [1] * 80)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=5008)

runs = []
for c_val in (0.05, 0.5, 1.0, 5.0):
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(C=c_val, max_iter=500))])
    pipe.fit(X_tr, y_tr)
    probs = pipe.predict_proba(X_te)[:, 1]
    runs.append({"C": c_val, "f1": round(float(f1_score(y_te, probs >= 0.5)), 4), "auc": round(float(roc_auc_score(y_te, probs)), 4)})

for r in runs:
    print(r)
assert max(r["auc"] for r in runs) > 0.90


{'C': 0.05, 'f1': 0.7429, 'auc': 0.9581}
{'C': 0.5, 'f1': 0.75, 'auc': 0.9575}
{'C': 1.0, 'f1': 0.7317, 'auc': 0.9575}
{'C': 5.0, 'f1': 0.7317, 'auc': 0.9569}
